# API wrappers

The OpenWeatherMap offers REST endpoints for querying current weather, forecasts, historical data, etc. However, accessing this data directly via the REST API requires handling multiple API calls, query parameters, and response parsing. The pyowm library abstracts these complexities and provides useful built-in functionalities.

After signing in to OpenWeatherMap retrieve your api key at https://home.openweathermap.org/api_keys

You will also need to install the pyowm package: pip install pyowm 

In [36]:
import requests
import pyowm
import json

api_key = '92568ff5dfe7d723b0911d800713df8e'

## use case 1: managing API keys

In a raw rest API call you always have to manage credentials in each individual call. Wrappers usually store and manage the authentication for you

In [39]:
#You can get current weather data by making a GET request to an endpoint like:

params = {
    'appid' : api_key
}

response = requests.get('https://api.openweathermap.org/data/2.5/weather?q=London', params = params)

json.loads(response.text)

#but for every call you make using GET from now on you do need to add the parameters, since the raw API does not manage authentication for you

{'coord': {'lon': -0.1257, 'lat': 51.5085},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04n'}],
 'base': 'stations',
 'main': {'temp': 282.88,
  'feels_like': 280.55,
  'temp_min': 281.86,
  'temp_max': 283.5,
  'pressure': 1036,
  'humidity': 88,
  'sea_level': 1036,
  'grnd_level': 1031},
 'visibility': 10000,
 'wind': {'speed': 4.63, 'deg': 30},
 'clouds': {'all': 100},
 'dt': 1731441039,
 'sys': {'type': 2,
  'id': 268730,
  'country': 'GB',
  'sunrise': 1731395639,
  'sunset': 1731428134},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

Most wrappers (pyowm included) include some way of initializing a session with the authentication key that you then don't need to type again.

Initialize pyowm with the default configuration. Thenopen the weather manager

Check out a snippet here: https://pyowm.readthedocs.io/en/latest/v3/code-recipes.html#weather_data

In [41]:
from pyowm.owm import OWM

owm = OWM(api_key)

# Access the Weather Manager
weather_manager = owm.weather_manager()

# Get current weather data for a city
city = 'London'
observation = weather_manager.weather_at_place(city)
weather = observation.weather

# Print weather details
print(f"City: {city}")
print(f"Temperature: {weather.temperature('celsius')['temp']}°C")
print(f"Status: {weather.status}")
print(f"Detailed Status: {weather.detailed_status}")
print(f"Wind Speed: {weather.wind()['speed']} m/s")
print(f"Humidity: {weather.humidity}%")
print(f"Pressure: {weather.pressure['press']} hPa")

City: London
Temperature: 9.72°C
Status: Clouds
Detailed Status: overcast clouds
Wind Speed: 4.63 m/s
Humidity: 88%
Pressure: 1036 hPa


## use case 2: Simplified calls

With the raw REST API, you'd have to build a URL manually, send the request, and parse the JSON response to get the current weather.

In [43]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'

response = requests.get(url,params= params)
data = response.json()
temperature = data['main']['temp']
humidity = data['main']['humidity']
wind_speed = data['wind']['speed']

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 282.88°C, Humidity: 88%, Wind Speed: 4.63 m/s


Get the equivalent call as above for the city of London using the pyowm package

In [45]:
owm = OWM(api_key)  # Initialize pyowm with your API key

# Access the Weather Manager
weather_manager = owm.weather_manager()

# Get current weather data for London
city = 'London'
observation = weather_manager.weather_at_place(city)
weather = observation.weather

# Retrieve and print the temperature, humidity, and wind speed
temperature = weather.temperature('celsius')['temp']  
humidity = weather.humidity  
wind_speed = weather.wind()['speed'] 

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 9.72°C, Humidity: 88%, Wind Speed: 4.63 m/s


## use case 3: Combining and chaining calls

Wrappers often offer methods that make multiple calls to batch requests that make sense to batch. And often they offer methods that make sequences of calls that each returns information necessary to make the next call.

For example, to get a weather forecast for a specific city using the raw API you need to first geocode the city to get its latitude and longitude:

In [47]:
city = 'New York'
geocode_url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'
geocode_response = requests.get(geocode_url,params=params).json()

lat = geocode_response['coord']['lat']
lon = geocode_response['coord']['lon']

Then, request the weather forecast for that latitude/longitude:

In [49]:
forecast_url = f'http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}'
forecast_response = requests.get(forecast_url, params=params).json()

for entry in forecast_response['list']:
    print(f"Time: {entry['dt_txt']}, Temp: {entry['main']['temp']}°C")

Time: 2024-11-12 21:00:00, Temp: 284.12°C
Time: 2024-11-13 00:00:00, Temp: 283.33°C
Time: 2024-11-13 03:00:00, Temp: 281.39°C
Time: 2024-11-13 06:00:00, Temp: 278.2°C
Time: 2024-11-13 09:00:00, Temp: 277.16°C
Time: 2024-11-13 12:00:00, Temp: 276.57°C
Time: 2024-11-13 15:00:00, Temp: 278.84°C
Time: 2024-11-13 18:00:00, Temp: 281.75°C
Time: 2024-11-13 21:00:00, Temp: 282.55°C
Time: 2024-11-14 00:00:00, Temp: 281.78°C
Time: 2024-11-14 03:00:00, Temp: 281.26°C
Time: 2024-11-14 06:00:00, Temp: 280.55°C
Time: 2024-11-14 09:00:00, Temp: 279.86°C
Time: 2024-11-14 12:00:00, Temp: 279.43°C
Time: 2024-11-14 15:00:00, Temp: 280.57°C
Time: 2024-11-14 18:00:00, Temp: 281.27°C
Time: 2024-11-14 21:00:00, Temp: 281.66°C
Time: 2024-11-15 00:00:00, Temp: 281.62°C
Time: 2024-11-15 03:00:00, Temp: 281.31°C
Time: 2024-11-15 06:00:00, Temp: 281.03°C
Time: 2024-11-15 09:00:00, Temp: 280.29°C
Time: 2024-11-15 12:00:00, Temp: 280.1°C
Time: 2024-11-15 15:00:00, Temp: 281.67°C
Time: 2024-11-15 18:00:00, Temp: 285

Two calls: one for geocoding, one for forecasts.
But with pyowm, because this is a common operation, there is a method that handles the geocoding internally and then fetches the weather forecast in one step.

Get the above forecast in a single call using pyowm.

Hint: search for "forecast_at_place" in the code recipies of the documentation

In [55]:
owm = OWM(api_key)  # Initialize pyowm with your API key

# Access the Weather Manager
weather_manager = owm.weather_manager()

# Get the 5-day forecast for New York
city = 'New York'
forecast = weather_manager.forecast_at_place(city, '3h') 

# Loop through forecast entries and print the time and temperature
for weather in forecast.forecast:
    time = weather.reference_time('date')
    temp = weather.temperature('celsius')['temp']
    print(f"Time: {time}, Temp: {temp}°C")

Time: 2024-11-12 21:00:00+00:00, Temp: 10.79°C
Time: 2024-11-13 00:00:00+00:00, Temp: 10.06°C
Time: 2024-11-13 03:00:00+00:00, Temp: 8.18°C
Time: 2024-11-13 06:00:00+00:00, Temp: 5.05°C
Time: 2024-11-13 09:00:00+00:00, Temp: 4.01°C
Time: 2024-11-13 12:00:00+00:00, Temp: 3.42°C
Time: 2024-11-13 15:00:00+00:00, Temp: 5.69°C
Time: 2024-11-13 18:00:00+00:00, Temp: 8.6°C
Time: 2024-11-13 21:00:00+00:00, Temp: 9.4°C
Time: 2024-11-14 00:00:00+00:00, Temp: 8.63°C
Time: 2024-11-14 03:00:00+00:00, Temp: 8.11°C
Time: 2024-11-14 06:00:00+00:00, Temp: 7.4°C
Time: 2024-11-14 09:00:00+00:00, Temp: 6.71°C
Time: 2024-11-14 12:00:00+00:00, Temp: 6.28°C
Time: 2024-11-14 15:00:00+00:00, Temp: 7.42°C
Time: 2024-11-14 18:00:00+00:00, Temp: 8.12°C
Time: 2024-11-14 21:00:00+00:00, Temp: 8.51°C
Time: 2024-11-15 00:00:00+00:00, Temp: 8.47°C
Time: 2024-11-15 03:00:00+00:00, Temp: 8.16°C
Time: 2024-11-15 06:00:00+00:00, Temp: 7.88°C
Time: 2024-11-15 09:00:00+00:00, Temp: 7.14°C
Time: 2024-11-15 12:00:00+00:00, Te

## use case 4: Convenience methods

Wrappers often offer built-in methods to handle common kinds of tasks related to the APIs, reducing the need for manual calculations.

for example converting units (e.g., temperature from Celsius to Fahrenheit) or working with more complex data requires manual conversion when using the raw API.

In [57]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}'

response = requests.get(url)
data = response.json()
temperature_celsius = data['main']['temp']
temperature_fahrenheit = (temperature_celsius * 9/5) + 32

print(f"Temperature in Celsius: {temperature_celsius}°C, Fahrenheit: {temperature_fahrenheit}°F")

Temperature in Celsius: 282.88°C, Fahrenheit: 541.184°F


But the pyowm wrapper offers built-in methods to handle these kinds of tasks, reducing the need for manual calculations.
Get the temperature both in Celcius and Farenheit using pyowm. Navigate the code recipes to figure out the inbuilt methods for this.

In [59]:
owm = OWM(api_key) 

# Access the Weather Manager
weather_manager = owm.weather_manager()

# Get current weather data for London
city = 'London'
observation = weather_manager.weather_at_place(city)
weather = observation.weather

# Retrieve temperatures in Celsius and Fahrenheit
temperature_celsius = weather.temperature('celsius')['temp']
temperature_fahrenheit = weather.temperature('fahrenheit')['temp']

print(f"Temperature in Celsius: {temperature_celsius}°C")
print(f"Temperature in Fahrenheit: {temperature_fahrenheit}°F")

Temperature in Celsius: 9.72°C
Temperature in Fahrenheit: 49.5°F
